# ⚗️ AfriOmics | Metabolomics Module
## Notebook 03 — From Raw Spectra to Metabolite Insights
---
**What this notebook does:**
1. Load and validate processed metabolite peak tables (from MZmine3/XCMS)
2. Normalise and filter metabolite features
3. PCA and group-level visualisation
4. Differential metabolite analysis (disease vs healthy)
5. Pathway enrichment mapping (KEGG/MetaCyc)
6. Microbiome-metabolite correlations (taxa → metabolite output)
7. African diet & metabolome contextualisation

> 💡 **Key concept:** Metabolomics captures the chemical output of your microbial community — the small molecules that actually interface with host biology, drive inflammation, modulate immunity, and are altered in infectious disease.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'matplotlib', 'seaborn',
                'plotly', 'scipy', 'scikit-learn'], check=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.spatial.distance import braycurtis
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

print('✅ AfriOmics Metabolomics Notebook v1.0')

In [ ]:
CONFIG = {
    'results_dir'    : 'results/metabolomics/',
    'metadata_file'  : 'config/samples.tsv',
    'group_col'      : 'disease',
    'ref_level'      : 'healthy',
    'fdr_cutoff'     : 0.05,
    'fc_cutoff'      : 1.5,
    'corr_threshold' : 0.4,
}
for d in ['stats', 'pathways', 'correlations', 'african_context']:
    Path(f"{CONFIG['results_dir']}/{d}").mkdir(parents=True, exist_ok=True)
print('✅ Configuration ready.')

---
## STEP 1 — Load & Normalise Metabolite Table
**Input:** Feature table from MZmine3 or XCMS — rows=metabolites, columns=samples, values=peak intensities.

**Normalisation options:**
- **Total ion current (TIC):** divide each sample by its total intensity
- **Median fold change:** robust to outlier metabolites
- **PQN (probabilistic quotient):** recommended for urine/faecal samples
- **Log transformation:** stabilises variance for statistical testing

In [ ]:
def normalise_metabolomics(peak_df, method='pqn', log_transform=True):
    """
    Normalise metabolite peak intensity table.

    Parameters:
    -----------
    peak_df       : metabolites × samples DataFrame
    method        : 'tic' | 'pqn' | 'median'
    log_transform : apply log2(x+1) after normalisation

    Returns:
    --------
    Normalised DataFrame
    """
    df = peak_df.copy().astype(float)
    df[df < 0] = 0  # Clip negatives

    if method == 'tic':
        # Total Ion Current: divide each sample by its column sum
        df = df.div(df.sum(axis=0), axis=1) * 1e6   # per million

    elif method == 'median':
        # Median fold change normalisation
        ref_medians = df.median(axis=1)
        factors = df.div(ref_medians + 1e-10, axis=0).median(axis=0)
        df = df.div(factors, axis=1)

    elif method == 'pqn':
        # Probabilistic Quotient Normalisation
        # Reference = median spectrum across all samples
        ref = df.median(axis=1)
        quotients = df.div(ref + 1e-10, axis=0)
        factors   = quotients.median(axis=0)
        df = df.div(factors, axis=1)

    # Impute zeros with half the minimum positive value
    min_val = df[df > 0].min().min() / 2
    df = df.fillna(0)
    df[df == 0] = min_val

    if log_transform:
        df = np.log2(df)

    print(f'Normalisation ({method}) complete:')
    print(f'  Metabolites: {df.shape[0]}')
    print(f'  Samples    : {df.shape[1]}')
    print(f'  Value range: [{df.min().min():.2f}, {df.max().max():.2f}]')
    return df


def filter_metabolites(df, min_prevalence=0.5, min_cv=0.1):
    """
    Filter metabolites:
    - Remove features detected in fewer than min_prevalence fraction of samples
    - Remove features with low coefficient of variation (uninformative)
    """
    n_before = df.shape[0]
    # Prevalence filter
    finite_df = df.replace([np.inf, -np.inf], np.nan)
    prev = finite_df.notna().sum(axis=1) / df.shape[1]
    df = df[prev >= min_prevalence]
    # CV filter
    cv  = df.std(axis=1) / (df.mean(axis=1).abs() + 1e-10)
    df  = df[cv >= min_cv]
    print(f'Filtered: {n_before} → {df.shape[0]} metabolites retained')
    return df

print('✅ Metabolomics normalisation functions ready.')

---
## STEP 2 — Differential Metabolite Analysis
**What:** Identify metabolites significantly different between disease groups and healthy controls.

**Method:** Wilcoxon rank-sum test (non-parametric, appropriate for small n) + Benjamini-Hochberg FDR correction.

**African-relevant outputs:** Changes in butyrate, bile acids, tryptophan metabolites, kynurenines — all implicated in African infectious diseases.

In [ ]:
from statsmodels.stats.multitest import multipletests

def differential_metabolites(norm_df, metadata_df, group_col='disease',
                               ref_group='healthy', fdr_cutoff=0.05, fc_cutoff=1.5):
    """
    Wilcoxon rank-sum test: each disease group vs healthy controls.
    Returns full results table with fold change, p-value, FDR.
    """
    common = list(set(norm_df.columns) & set(metadata_df.index))
    meta   = metadata_df.loc[common]
    ref_s  = meta[meta[group_col] == ref_group].index.tolist()
    groups = [g for g in meta[group_col].unique() if g != ref_group]

    all_results = []
    for grp in groups:
        grp_s = meta[meta[group_col] == grp].index.tolist()
        rows  = []
        for met in norm_df.index:
            x = norm_df.loc[met, grp_s].dropna().values
            y = norm_df.loc[met, ref_s].dropna().values
            if len(x) < 2 or len(y) < 2:
                continue
            _, p = stats.mannwhitneyu(x, y, alternative='two-sided')
            fc   = np.mean(x) - np.mean(y)   # Log2 fold change (data already log-transformed)
            rows.append({'metabolite': met, 'comparison': f'{grp}_vs_{ref_group}',
                         'group': grp, 'log2fc': round(fc, 4), 'pvalue': p,
                         'mean_disease': round(np.mean(x), 4),
                         'mean_healthy': round(np.mean(y), 4)})

        res_df = pd.DataFrame(rows)
        if res_df.empty: continue
        _, fdr, _, _ = multipletests(res_df['pvalue'], method='fdr_bh')
        res_df['fdr'] = fdr
        res_df['significant'] = (res_df['fdr'] < fdr_cutoff) & (res_df['log2fc'].abs() >= np.log2(fc_cutoff))
        res_df['direction']   = np.where(res_df['log2fc'] > 0, 'UP', 'DOWN')
        all_results.append(res_df)

        n_sig = res_df['significant'].sum()
        print(f'  {grp} vs {ref_group}: {n_sig} significant metabolites (FDR<{fdr_cutoff}, FC>{fc_cutoff})')

    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


def plot_volcano(diff_df, comparison, fdr_cut=0.05, fc_cut=1.5):
    """Volcano plot for one comparison."""
    d = diff_df[diff_df['comparison'] == comparison].copy()
    d['-log10fdr'] = -np.log10(d['fdr'].clip(lower=1e-10))
    d['colour']    = 'NS'
    d.loc[(d['fdr'] < fdr_cut) & (d['log2fc'] >  np.log2(fc_cut)), 'colour'] = 'UP'
    d.loc[(d['fdr'] < fdr_cut) & (d['log2fc'] < -np.log2(fc_cut)), 'colour'] = 'DOWN'

    colour_map = {'UP': '#ff6b6b', 'DOWN': '#5bc4ff', 'NS': '#3a4a3a'}

    fig = px.scatter(
        d, x='log2fc', y='-log10fdr', color='colour',
        color_discrete_map = colour_map,
        hover_name = 'metabolite',
        title  = f'Differential Metabolites: {comparison}<br><sup>AfriOmics Metabolomics</sup>',
        labels = {'log2fc': 'Log2 Fold Change', '-log10fdr': '-Log10 FDR'},
        template = 'plotly_dark',
        opacity = 0.8
    )
    fig.add_hline(y=-np.log10(fdr_cut), line_dash='dash', line_color='#f5c842')
    fig.add_vline(x= np.log2(fc_cut),  line_dash='dash', line_color='#f5c842')
    fig.add_vline(x=-np.log2(fc_cut),  line_dash='dash', line_color='#f5c842')
    fig.update_layout(paper_bgcolor='#111710', plot_bgcolor='#111710', font_color='#e8f0e9')
    return fig

print('✅ Differential metabolite functions ready.')

---
## STEP 3 — African Diet & Metabolome Context
**What:** Tag metabolites that are known to be influenced by African dietary patterns.

**Why:** Without this step, diet-driven metabolite differences could be misattributed to disease.

**African dietary metabolite categories:**
- **Fermented food markers:** lactate, acetate, ethanol derivatives (ogi, togwa, mahewu, injera)
- **SCFA:** butyrate, propionate, acetate (high-fibre traditional diet)
- **Plant polyphenols:** from moringa, baobab, shea, rooibos
- **Phytate metabolites:** from millet, sorghum, cassava staples
- **Tryptophan pathway:** altered by both malaria and dietary tryptophan content

In [ ]:
# African dietary metabolite context database (curated)
AFRICAN_DIET_METABOLITES = {
    'SCFA': {
        'metabolites': ['butyrate', 'propionate', 'acetate', 'valerate',
                         'isobutyrate', 'isovalerate'],
        'context': 'Short-chain fatty acids — elevated in high-fibre traditional African diets',
        'dietary_source': 'Fermented cereals, legumes, root vegetables',
        'disease_relevance': 'Reduced in malaria, HIV gut dysbiosis; protective against colonisation'
    },
    'Fermented_food_markers': {
        'metabolites': ['lactate', 'succinate', 'acetoin', 'diacetyl', 'formate'],
        'context': 'Microbial fermentation products from traditional African fermented foods',
        'dietary_source': 'Ogi (West Africa), togwa (East Africa), mahewu (Southern Africa), injera (Ethiopia)',
        'disease_relevance': 'Marker of traditional vs western diet; confounds disease studies'
    },
    'Polyphenol_metabolites': {
        'metabolites': ['quercetin', 'kaempferol', 'rutin', 'chlorogenic_acid',
                         'gallic_acid', 'catechin', 'epicatechin'],
        'context': 'Plant polyphenol metabolites from African medicinal plants and foods',
        'dietary_source': 'Moringa, baobab, rooibos, shea, African leafy vegetables',
        'disease_relevance': 'Anti-inflammatory; can mask host immune metabolites'
    },
    'Tryptophan_kynurenine': {
        'metabolites': ['tryptophan', 'kynurenine', 'kynurenic_acid',
                         'quinolinic_acid', 'serotonin', 'indole'],
        'context': 'Tryptophan metabolism pathway — altered in malaria and TB',
        'dietary_source': 'Dietary tryptophan (animal protein, legumes)',
        'disease_relevance': 'KEY malaria biomarker: P.falciparum sequesters tryptophan; kynurenine ratio elevated'
    },
    'Bile_acids': {
        'metabolites': ['cholic_acid', 'deoxycholic_acid', 'lithocholic_acid',
                         'ursodeoxycholic_acid', 'chenodeoxycholic_acid'],
        'context': 'Secondary bile acids produced by gut bacteria from host primary bile acids',
        'dietary_source': 'Dietary fat content',
        'disease_relevance': 'Altered in HIV; protective at high levels against C.diff (less relevant in Africa)'
    },
    'Phytate_cereal_markers': {
        'metabolites': ['phytate', 'inositol', 'phosphate'],
        'context': 'High in millet, sorghum, cassava — staple African crops',
        'dietary_source': 'Millet, sorghum, cassava, maize',
        'disease_relevance': 'Mineral chelation; reduces iron/zinc absorption; important confounder'
    }
}


def tag_african_dietary_metabolites(diff_df, annotations_df=None):
    """
    Tag differential metabolites with African dietary context.
    Helps identify which differences are likely diet-driven vs disease-driven.
    """
    diff_df = diff_df.copy()
    diff_df['african_dietary_category'] = 'Unknown'
    diff_df['african_context']          = ''
    diff_df['dietary_source']           = ''
    diff_df['disease_relevance']        = ''

    for category, info in AFRICAN_DIET_METABOLITES.items():
        for met_keyword in info['metabolites']:
            mask = diff_df['metabolite'].str.lower().str.contains(met_keyword.lower(), na=False)
            diff_df.loc[mask, 'african_dietary_category'] = category
            diff_df.loc[mask, 'african_context']          = info['context']
            diff_df.loc[mask, 'dietary_source']           = info['dietary_source']
            diff_df.loc[mask, 'disease_relevance']        = info['disease_relevance']

    # Summary
    cat_counts = diff_df[diff_df['significant'] == True]['african_dietary_category'].value_counts()
    print('African dietary category breakdown (significant metabolites):')
    print(cat_counts.to_string())
    print(f'\nUntagged (Unknown): {cat_counts.get("Unknown", 0)} — may be novel or disease-specific')

    return diff_df

print('✅ African dietary context database loaded.')
print(f'   {len(AFRICAN_DIET_METABOLITES)} categories with {sum(len(v["metabolites"]) for v in AFRICAN_DIET_METABOLITES.values())} metabolite keywords')

In [ ]:
print('=' * 60)
print('AfriOmics | Metabolomics Demo Run')
print('=' * 60)

np.random.seed(42)
N = 24
M = 80
sample_names = [f'AFRI_{str(i).zfill(3)}' for i in range(1, N+1)]

met_names = (
    ['butyrate', 'propionate', 'acetate', 'tryptophan',
     'kynurenine', 'kynurenic_acid', 'cholic_acid', 'deoxycholic_acid',
     'lactate', 'succinate', 'quercetin', 'kaempferol'] +
    [f'Metabolite_{i}' for i in range(M - 12)]
)

metadata_demo = pd.DataFrame({
    'disease'    : ['malaria']*8 + ['HIV']*8 + ['healthy']*8,
    'region'     : ['west_africa']*8 + ['east_africa']*8 + ['southern_africa']*8,
    'diet_type'  : np.random.choice(['traditional','mixed','western'], N)
}, index=sample_names)

# Raw peak intensities
raw_df = pd.DataFrame(
    np.random.lognormal(10, 2, (M, N)),
    index=met_names, columns=sample_names
)

# Inject disease signals
mal = metadata_demo[metadata_demo['disease']=='malaria'].index.tolist()
hiv = metadata_demo[metadata_demo['disease']=='HIV'].index.tolist()
raw_df.loc['butyrate', mal]       *= 0.15   # Depleted in malaria
raw_df.loc['kynurenine', mal]     *= 6.0    # Elevated in malaria
raw_df.loc['tryptophan', mal]     *= 0.3    # Depleted (consumed by parasite)
raw_df.loc['cholic_acid', hiv]    *= 0.4    # Bile acid disruption in HIV
raw_df.loc['lactate', mal]        *= 3.0    # Anaerobic metabolism

# Normalise
print('\n[1/3] Normalising metabolomics data (PQN + log2)...')
norm_df = normalise_metabolomics(raw_df, method='pqn', log_transform=True)
norm_df = filter_metabolites(norm_df, min_prevalence=0.8)

# PCA
print('\n[2/3] Running PCA...')
X_t = norm_df.T.fillna(0)
scaler = StandardScaler()
X_s = scaler.fit_transform(X_t)
pca = PCA(n_components=5, random_state=42)
coords = pca.fit_transform(X_s)
var_exp = np.round(pca.explained_variance_ratio_ * 100, 1)
pca_df = pd.DataFrame(coords[:, :3], columns=['PC1','PC2','PC3'], index=sample_names)
pca_df = pca_df.join(metadata_demo)

fig_pca = px.scatter(
    pca_df.reset_index(), x='PC1', y='PC2', color='disease',
    hover_name='index',
    title=f'Metabolomics PCA<br><sup>PC1: {var_exp[0]}% | PC2: {var_exp[1]}%</sup>',
    template='plotly_dark',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig_pca.update_traces(marker=dict(size=12, opacity=0.85))
fig_pca.update_layout(paper_bgcolor='#111710', plot_bgcolor='#111710', font_color='#e8f0e9')
fig_pca.show()

# Differential analysis
print('\n[3/3] Differential metabolite analysis...')
diff_df = differential_metabolites(norm_df, metadata_demo, fdr_cutoff=0.2, fc_cutoff=1.3)

# Tag African dietary context
diff_df = tag_african_dietary_metabolites(diff_df)

# Volcano plot for malaria
comparisons = diff_df['comparison'].unique()
for comp in comparisons:
    fig_vol = plot_volcano(diff_df, comp, fdr_cut=0.2, fc_cut=1.3)
    fig_vol.show()

# Save
norm_df.to_csv(f"{CONFIG['results_dir']}/stats/normalised_metabolites.tsv", sep='\t')
diff_df.to_csv(f"{CONFIG['results_dir']}/stats/differential_metabolites.tsv", sep='\t', index=False)
pca_df.to_csv(f"{CONFIG['results_dir']}/stats/pca_coordinates.tsv", sep='\t')

print('\n✅ Metabolomics demo complete. Results saved to results/metabolomics/')